# NFHS-5 State-Level Health Indicator Analysis
Data source: Kaggle — `archive.zip` containing `National Family Health Survey (NFHS-5) 2019-20.csv`

## Setup

In [ ]:
# Install/upgrade required packages in Colab
!pip install -q folium pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import requests
import zipfile
import os

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

## Phase 1: Data Setup and Aggregation

In [ ]:
# ---------------------------------------------------------
# Step 1: Upload and extract the Kaggle dataset (archive.zip)
# ---------------------------------------------------------
from google.colab import files
uploaded = files.upload()  # select archive.zip from your Kaggle download

zip_filename = list(uploaded.keys())[0]
extract_dir = "nfhs5_kaggle_data"

with zipfile.ZipFile(zip_filename, "r") as zf:
    zf.extractall(extract_dir)

csv_files = [f for f in os.listdir(extract_dir) if f.lower().endswith(".csv")]
print("CSV files found:", csv_files)

csv_path = os.path.join(extract_dir, csv_files[0])
raw_df = pd.read_csv(csv_path)

print(raw_df.shape)
print(raw_df.columns.tolist())
raw_df.head()

In [ ]:
# ---------------------------------------------------------
# Step 2: Filter to key health columns and pivot long -> wide
# ---------------------------------------------------------
# This Kaggle file is in LONG format with a messy 2-row header:
# row 0 of the data is actually the Urban/Rural/Total sub-header for the
# 'NFHS-5 (2019-20)' column group, so it is dropped after renaming columns.
raw_df.columns = [
    "s_no", "indicator_code", "category", "sub_indicator",
    "nfhs5_urban", "nfhs5_rural", "nfhs5_total", "nfhs4_total", "state",
]
raw_df = raw_df.iloc[1:].reset_index(drop=True)

# Exact sub_indicator strings verified directly from this file —
# this dataset (unlike some other NFHS-5 sources) explicitly suffixes
# blood sugar and hypertension rows with '- Women' / '- Men', so there is
# no ambiguity about which gender each value belongs to.
indicator_map = {
    "Women who are overweight or obese (BMI ≥25.0 kg/m2) (%)": "Obesity_Women",
    "Men who are overweight or obese (BMI ≥25.0 kg/m2) (%)": "Obesity_Men",
    "Children age 6-59 months who are anaemic (<11.0 g/dl) (%)": "Anaemia_Children",
    "All women age 15-49 years who are anaemic (%)": "Anaemia_Women",
    "Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Women": "BloodSugar_Women",
    "Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%) - Men": "BloodSugar_Men",
    "Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%) - Women": "Hypertension_Women",
    "Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%) - Men": "Hypertension_Men",
}

subset = raw_df[raw_df["sub_indicator"].isin(indicator_map.keys())].copy()
subset["clean_col"] = subset["sub_indicator"].map(indicator_map)

print(f"Matched {subset['sub_indicator'].nunique()} of {len(indicator_map)} target indicators")

# Sanity check: each indicator should appear exactly once per state
dup_check = subset.groupby(["state", "clean_col"]).size()
assert (dup_check == 1).all(), "Duplicate (state, indicator) pairs found — inspect raw_df before proceeding."

# Pivot from long to wide format using the national-total column (nfhs5_total)
df = subset.pivot_table(
    index="state", columns="clean_col", values="nfhs5_total", aggfunc="first"
).reset_index().rename(columns={"state": "State"})

# Drop the national aggregate row — keep only states/UTs
df = df[df["State"] != "India"].reset_index(drop=True)

print(df.shape)
df.head()

In [ ]:
# ---------------------------------------------------------
# Step 3: Clean non-numeric placeholders, cast to float
# ---------------------------------------------------------
# This source uses several placeholder styles seen during inspection:
#   - thousands separators, e.g. '1,020'
#   - parenthesized values for small unweighted samples, e.g. '(91.7)' (still numeric)
#   - '*' or 'na' for genuinely missing/suppressed values
numeric_cols = [c for c in df.columns if c != "State"]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)   # thousands separator
        .str.replace("(", "", regex=False)    # parenthesized small-sample flag
        .str.replace(")", "", regex=False)
        .str.replace("-", "", regex=False)    # dashboard hyphen placeholder
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["State"]).reset_index(drop=True)

print(df.isna().sum())
df.head()

In [ ]:
# ---------------------------------------------------------
# Step 4: Gender_Obesity_Gap
# ---------------------------------------------------------
df["Gender_Obesity_Gap"] = df["Obesity_Women"] - df["Obesity_Men"]

df[["State", "Obesity_Men", "Obesity_Women", "Gender_Obesity_Gap"]] \
    .sort_values("Gender_Obesity_Gap", ascending=False).head(10)

## Phase 2: Exploratory Data Analysis (EDA) and Plotting

In [ ]:
# ---------------------------------------------------------
# Task 1: Correlation matrix — double burden of malnutrition
# ---------------------------------------------------------
corr_cols = ["Anaemia_Children", "Obesity_Men", "Obesity_Women"]
corr_df = df[corr_cols].dropna()

corr_matrix = corr_df.corr(method="pearson")
print(corr_matrix)

plt.figure(figsize=(6, 5))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, square=True, cbar_kws={"label": "Pearson r"}
)
plt.title("Correlation: Childhood Anaemia vs Adult Obesity")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Task 2: Grouped bar chart — Top 10 hypertension states
# ---------------------------------------------------------
df["Hypertension_Avg"] = df[["Hypertension_Men", "Hypertension_Women"]].mean(axis=1)
top10 = df.nlargest(10, "Hypertension_Avg")

x = np.arange(len(top10))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width/2, top10["Hypertension_Men"], width, label="Men", color="#1f77b4")
ax.bar(x + width/2, top10["Hypertension_Women"], width, label="Women", color="#d62728")

ax.set_xticks(x)
ax.set_xticklabels(top10["State"], rotation=45, ha="right")
ax.set_ylabel("Hypertension Prevalence (%)")
ax.set_title("Top 10 States: Hypertension by Gender (NFHS-5)")
ax.legend(title="Gender")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Task 3: Scatter plot — Anaemic Women vs Obese Women
# ---------------------------------------------------------
state_abbr = {
    "Andhra Pradesh": "AP", "Arunachal Pradesh": "AR", "Assam": "AS",
    "Bihar": "BR", "Chhattisgarh": "CG", "Goa": "GA", "Gujarat": "GJ",
    "Haryana": "HR", "Himachal Pradesh": "HP", "Jharkhand": "JH",
    "Karnataka": "KA", "Kerala": "KL", "Madhya Pradesh": "MP",
    "Maharashtra": "MH", "Manipur": "MN", "Meghalaya": "ML",
    "Mizoram": "MZ", "Nagaland": "NL", "Odisha": "OD", "Punjab": "PB",
    "Rajasthan": "RJ", "Sikkim": "SK", "Tamil Nadu": "TN",
    "Telangana": "TG", "Tripura": "TR", "Uttar Pradesh": "UP",
    "Uttarakhand": "UK", "West Bengal": "WB", "NCT Delhi": "DL",
    "Jammu & Kashmir": "JK", "Ladakh": "LA", "Puducherry": "PY",
}

scatter_df = df.dropna(subset=["Anaemia_Women", "Obesity_Women"]).copy()
scatter_df["Abbr"] = scatter_df["State"].map(state_abbr).fillna(
    scatter_df["State"].str[:2].str.upper()
)

plt.figure(figsize=(9, 7))
plt.scatter(
    scatter_df["Anaemia_Women"], scatter_df["Obesity_Women"],
    s=80, c="#2ca02c", edgecolor="black", alpha=0.8
)

for _, row in scatter_df.iterrows():
    plt.annotate(
        row["Abbr"],
        (row["Anaemia_Women"], row["Obesity_Women"]),
        textcoords="offset points", xytext=(5, 5), fontsize=9
    )

plt.xlabel("Anaemic Women (%)")
plt.ylabel("Obese Women (%)")
plt.title("Anaemia vs Obesity in Women — State-wise (NFHS-5)")
plt.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## Phase 3: Geographic Visualization (Folium)

In [ ]:
# ---------------------------------------------------------
# Step 1: Initialize base map of India
# ---------------------------------------------------------
india_map = folium.Map(location=[20.5937, 78.9629], zoom_start=5, tiles="cartodbpositron")
india_map

In [ ]:
# ---------------------------------------------------------
# Step 2: Download GeoJSON, bind via Choropleth
# ---------------------------------------------------------
geojson_url = "https://raw.githubusercontent.com/geohacker/india/master/state/india_telengana.geojson"
geojson_data = requests.get(geojson_url).json()

geo_state_names = [feat["properties"].get("NAME_1") for feat in geojson_data["features"]]
print(sorted(set(geo_state_names)))

In [ ]:
# Compare names and find mismatches between df["State"] and the GeoJSON
df_states = set(df["State"].unique())
geo_states = set(geo_state_names)

print("In data but not in GeoJSON:", df_states - geo_states)
print("In GeoJSON but not in data:", geo_states - df_states)

In [ ]:
# Fix mismatches found above (verified against this specific GeoJSON source).
# Ladakh has no separate match in this GeoJSON (it predates the 2019 J&K/Ladakh
# split) — it stays unmatched and renders as the 'nan_fill_color' (grey).
name_fix = {
    "Odisha": "Orissa",
    "Uttarakhand": "Uttaranchal",
    "NCT Delhi": "Delhi",
    "Jammu & Kashmir": "Jammu and Kashmir",
    "Andaman & Nicobar Islands": "Andaman and Nicobar",
    "Dadra & Nagar Haveli and Daman & Diu": "Dadra and Nagar Haveli",
}

df["State_geo"] = df["State"].replace(name_fix)

choropleth = folium.Choropleth(
    geo_data=geojson_data,
    name="Obesity Choropleth",
    data=df,
    columns=["State_geo", "Obesity_Women"],
    key_on="feature.properties.NAME_1",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Obesity in Women (%)",
    nan_fill_color="grey",
).add_to(india_map)

india_map

In [ ]:
# ---------------------------------------------------------
# Step 3: CircleMarkers with summary card tooltips at state capitals
# ---------------------------------------------------------
state_capitals = {
    "Andhra Pradesh": (16.5062, 80.6480), "Arunachal Pradesh": (27.0844, 93.6053),
    "Assam": (26.2006, 92.9376), "Bihar": (25.5941, 85.1376),
    "Chhattisgarh": (21.2787, 81.8661), "Goa": (15.4909, 73.8278),
    "Gujarat": (23.2156, 72.6369), "Haryana": (30.7333, 76.7794),
    "Himachal Pradesh": (31.1048, 77.1734), "Jharkhand": (23.6102, 85.2799),
    "Karnataka": (12.9716, 77.5946), "Kerala": (8.5241, 76.9366),
    "Madhya Pradesh": (23.2599, 77.4126), "Maharashtra": (19.0760, 72.8777),
    "Manipur": (24.8170, 93.9368), "Meghalaya": (25.5788, 91.8933),
    "Mizoram": (23.7271, 92.7176), "Nagaland": (25.6751, 94.1086),
    "Odisha": (20.2961, 85.8245), "Punjab": (30.7333, 76.7794),
    "Rajasthan": (26.9124, 75.7873), "Sikkim": (27.3389, 88.6065),
    "Tamil Nadu": (13.0827, 80.2707), "Telangana": (17.3850, 78.4867),
    "Tripura": (23.8315, 91.2868), "Uttar Pradesh": (26.8467, 80.9462),
    "Uttarakhand": (30.3165, 78.0322), "West Bengal": (22.5726, 88.3639),
    "NCT Delhi": (28.7041, 77.1025), "Jammu & Kashmir": (34.0837, 74.7973),
    "Puducherry": (11.9416, 79.8083), "Ladakh": (34.1526, 77.5770),
}

marker_layer = folium.FeatureGroup(name="State Health Summary")

for _, row in df.iterrows():
    coords = state_capitals.get(row["State"])
    if coords is None or pd.isna(row.get("BloodSugar_Women")):
        continue

    popup_html = f"""
    <div style="font-family: Arial; font-size: 13px;">
        <b>{row['State']}</b><br>
        High Blood Sugar: {row.get('BloodSugar_Women', 'N/A')}%<br>
        Anaemia: {row.get('Anaemia_Women', 'N/A')}%
    </div>
    """

    folium.CircleMarker(
        location=coords,
        radius=7,
        color="darkred",
        fill=True,
        fill_color="orange",
        fill_opacity=0.85,
        tooltip=folium.Tooltip(popup_html),
    ).add_to(marker_layer)

marker_layer.add_to(india_map)
folium.LayerControl().add_to(india_map)

india_map

In [ ]:
# Save final dashboard as a standalone HTML file
india_map.save("nfhs5_health_dashboard.html")
print("Saved: nfhs5_health_dashboard.html")